# The Richards Equation
_1-D unsaturated water flow_  
  
Caoimhe Mannion, November 2026  


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.interpolate import RectBivariateSpline as rbs
import sympy as sp

In [6]:
h, t, z = sp.symbols("h t z")

In [3]:
# example values, to update from Rawls 1982
theta_r = 0.1 #residual volumetric water content as K -> 0
theta_s = 0.5 #volumetric water content at saturation
lambda_ = 0.9 #pore size distribution index
K_s = 0.01 #cm/s

psi_b = None
h_b = 20 #cm

### Mixed Form
$$\frac{\partial \theta}{\partial t}=\frac{\partial}{\partial z}\left[K(h)\left(\frac{\partial h}{\partial z}+1 \right)\right]

### Water Content Form
$$\frac{\partial \theta}{\partial t}=\frac{\partial}{\partial z}\left[D_w(\theta)\left(\frac{\partial \theta}{\partial z}\right)+\frac{\partial K(\theta)}{\partial z}\right]

#### Functions
Brooks-Corey, Burdine Model
$$ S_e = \frac{\theta-\theta_r}{\theta_s-\theta_r} = \left(\frac{\psi_b}{\psi_m}\right)^{\lambda} = \left(\frac{h_b}{|h|}\right)^{\lambda} $$

In [ ]:
### Mixed Form ### 
h = np.linspace(-10e-5,0,30) #matric potential head, centimetres, negative for unsaturated soil, 0 at saturation.

def S_e(h):
    """Brooks-Corey Effective Saturation"""
    if abs(h) >= h_b:
        # saturation
        return 1
    else:
        return (h_b/abs(h))**(lambda_)

def theta(h):
    """Water Retention Curve"""
    return S_e(h)*(theta_s - theta_r) + theta_r

def K(h):
    """Burdine Hydraulic Conductivity"""
    return K_s*S_e(h)**(3+2/lambda_)

In [11]:
### Water Content Form ###
theta_arr = np.linspace(theta_r,theta_s,30) #volumetric water content, dimensionless, fraction in [0,1], defined strictly between theta_r,theta_s (functions are for unsaturated soil)

def K(theta):
    """Burdine Hydraulic Conductivity"""
    return K_s*S_e(theta)**(3+2/lambda_)

def S_e(theta):
    """Brooks-Corey Effective Saturation"""
    return (theta-theta_r)/(theta_s-theta_r)

def D(theta):
    """Diffusivity"""
    return K(theta)*h_b*(lambda_(theta_s-theta_r))**(-1)*S_e(theta)**(-1/lambda_ - 1)

In [7]:
def rich_mxd(h):
    return sp.diff(K(h)*(sp.diff(h,z)+1),z)